# ce-test
Generated from `code/business_entity_resolution/src/cross_encoder_score.py` by `jobs/build_notebook.py`. Edit the script, not this notebook.

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 anyascii==0.3.3 scipy==1.18.1 scikit-learn==1.9.1 sparse-dot-topn==1.2.0 lightgbm==4.7.0

In [ ]:
import os
os.makedirs('/tmp/src', exist_ok=True)
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, dirs, files)

In [ ]:
%%writefile /tmp/src/selection.py
"""Match selection: from scored candidate pairs to the predicted matches per S1, tuned for macro F0.5.

Rules (all tuned on validation, never on test):
- threshold: keep a pair only if its match probability p >= tau;
- one owner: every S2/S3 record matches at most one S1 (EDA: 0 of 7.6M train records match two), so each
  candidate is kept only for the S1 that gives it the highest p;
- margin: keep a pair only if p >= margin * (best p of that S1), which drops weak extra matches next to a strong one.
F0.5 weighs precision twice, so a wrong match costs more than a missed one: the tuned threshold is usually high.

f05_table() is a vectorised version of metric.py (same definition, checked against it in the tests and in
run_model.py): macro F0.5 over ALL evaluated S1, an S1 without predictions scores 1.0 if it has no true match.
"""
from __future__ import annotations

import itertools

import polars as pl

TAUS = [round(0.05 * i, 2) for i in range(2, 20)]  # 0.10 .. 0.95
MARGINS = [0.0, 0.3, 0.5, 0.7]
BETA2 = 0.25


def select(scored: pl.DataFrame, tau: float, margin: float = 0.0, one_owner: bool = True, tau2: float | None = None,
           sim: float = 0.9, owner_delta: float = 0.0) -> pl.DataFrame:
    """scored: s1, cand, p (+ s_sim_name, s_num_eq for the secondary rule). Returns the kept (s1, cand, p) rows.
    tau2: also keep a pair with tau2 <= p < tau if it strongly resembles a confident candidate of the same S1
    (s_sim_name >= sim and the same first house number). owner_delta: with one_owner, an S2/S3 claimed by two S1
    goes to the higher p only if it leads by more than owner_delta; otherwise it is dropped from both."""
    keep = pl.col("p") >= tau
    if tau2 is not None and tau2 < tau:
        keep = keep | ((pl.col("p") >= tau2) & (pl.col("s_sim_name") >= sim) & (pl.col("s_num_eq") >= 1)).fill_null(False)
    x = scored.filter(keep)
    if one_owner:  # ties: the smaller s1 id wins (deterministic)
        x = x.sort(["cand", "p", "s1"], descending=[False, True, False])
        if owner_delta > 0:
            lead = pl.col("p") - pl.col("p").shift(-1).over("cand")
            x = x.with_columns(pl.int_range(pl.len()).over("cand").alias("_r"), lead.fill_null(1.0).alias("_lead"))
            x = x.filter((pl.col("_r") == 0) & (pl.col("_lead") > owner_delta)).drop("_r", "_lead")
        else:
            x = x.unique("cand", keep="first", maintain_order=True)
    if margin > 0:
        x = x.filter(pl.col("p") >= margin * pl.col("p").max().over("s1"))
    return x.select("s1", "cand", "p")


def f05_table(pred: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame) -> pl.DataFrame:
    """Per evaluated S1: n_pred, n_true, tp, precision, recall, f05. pred / truth: s1, cand; s1s: s1 (+ any
    attribute columns, e.g. country), the S1 to evaluate. S1 missing from pred count as empty predictions."""
    keys = s1s.select("s1")
    pred = pred.join(keys, on="s1", how="semi").select("s1", "cand").unique()
    truth = truth.join(keys, on="s1", how="semi").select("s1", "cand").unique()
    tp = pred.join(truth, on=["s1", "cand"], how="semi").group_by("s1").len("tp")
    x = (s1s.join(pred.group_by("s1").len("n_pred"), on="s1", how="left")
         .join(truth.group_by("s1").len("n_true"), on="s1", how="left")
         .join(tp, on="s1", how="left")
         .with_columns(pl.col("n_pred", "n_true", "tp").fill_null(0)))
    prec = pl.when(pl.col("n_pred") > 0).then(pl.col("tp") / pl.col("n_pred"))
    rec = pl.when(pl.col("n_true") > 0).then(pl.col("tp") / pl.col("n_true"))
    f = (pl.when(pl.col("n_true") == 0).then((pl.col("n_pred") == 0).cast(pl.Float64))
         .when(pl.col("tp") == 0).then(0.0)
         .otherwise((1 + BETA2) * prec * rec / (BETA2 * prec + rec)))
    return x.with_columns(prec.alias("precision"), rec.alias("recall"), f.alias("f05"))


def summary(tab: pl.DataFrame, by: str | None = None) -> pl.DataFrame:
    """Macro F0.5, mean precision / recall (over S1 where defined), per group and overall (+ singletons)."""
    aggs = [pl.len().alias("s1"), pl.col("f05").mean().alias("macro_f05"), pl.col("precision").mean().alias("precision"),
            pl.col("recall").mean().alias("recall"), pl.col("n_pred").mean().alias("pred_per_s1")]
    kind = pl.when(pl.col("n_true") == 0).then(pl.lit("singleton")).otherwise(pl.lit("non_singleton"))
    parts = [tab.select(pl.lit("overall").alias("group"), *aggs),
             tab.with_columns(kind.alias("_k")).group_by("_k").agg(aggs).sort("_k")
             .select(pl.format("kind={}", "_k").alias("group"), *[a.meta.output_name() for a in aggs])]
    if by:
        parts.insert(1, tab.group_by(by).agg(aggs).sort(by).select(pl.format(f"{by}={{}}", by).alias("group"),
                                                                     *[a.meta.output_name() for a in aggs]))
    return pl.concat(parts, how="vertical_relaxed")


def tune(scored: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame, taus=TAUS, margins=MARGINS,
         owners=(True, False)) -> pl.DataFrame:
    """Macro F0.5 on s1s for every (tau, margin, one_owner). Sorted best first."""
    rows = []
    for tau, margin, owner in itertools.product(taus, margins, owners):
        tab = f05_table(select(scored, tau, margin, owner), truth, s1s)
        rows.append({"tau": tau, "margin": margin, "one_owner": owner, "macro_f05": tab["f05"].mean(),
                     "pred_per_s1": tab["n_pred"].mean()})
    return pl.DataFrame(rows).sort("macro_f05", descending=True)


In [ ]:
%%writefile /tmp/src/submission.py
"""Write matching_results.tsv / candidate_pairs.tsv in the exact challenge format.

One row per S1 (in the given order), tab-separated, comma-joined IDs, no quoting,
no duplicates, empty second column when there is nothing to list.
"""
from collections.abc import Iterable, Mapping
from pathlib import Path

MATCHING_HEADER = ("source1_entity_id", "matched_entity_ids")
CANDIDATE_HEADER = ("source1_entity_id", "candidate_entity_ids")


def write_id_lists(path: Path, header: tuple[str, str], s1_ids: Iterable[str],
                   lists: Mapping[str, Iterable[str]]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\t".join(header) + "\n")
        for s1 in s1_ids:
            ids = list(dict.fromkeys(i for i in lists.get(s1, ()) if not i.startswith("S1-")))
            f.write(f"{s1}\t{','.join(ids)}\n")
            n += 1
    return n


def write_submission(out_dir: Path, s1_ids: list[str], matches: Mapping[str, Iterable[str]],
                     candidates: Mapping[str, Iterable[str]]) -> None:
    for s1, m in matches.items():
        extra = set(m) - set(candidates.get(s1, ()))
        if extra:
            raise ValueError(f"{s1}: matches not in candidates: {sorted(extra)[:5]}")
    write_id_lists(out_dir / "matching_results.tsv", MATCHING_HEADER, s1_ids, matches)
    write_id_lists(out_dir / "candidate_pairs.tsv", CANDIDATE_HEADER, s1_ids, candidates)


In [ ]:
%%writefile /tmp/src/validate_submission.py
#!/usr/bin/env python3
"""
ML Challenge 2026 — Submission Validator

Run this BEFORE submitting. It checks your output files against every formatting
rule the scorer enforces, so you can catch a rejection locally instead of burning
a submission. It reads only your output files and the test source files (to learn
which S1 entities are required and which S2/S3 IDs exist); it never needs the
ground truth and never computes your score.

It validates two files:

* ``matching_results.tsv`` (required) — your final matches, the file scored on the
  leaderboard.
* ``candidate_pairs.tsv`` (optional) — the candidate set from your blocking stage.
  When present, the validator also checks that your final matches are a subset of
  your candidates and *warns* (never fails) otherwise. When absent it is skipped
  with a warning; it is still expected in your final submission zip.

Stdlib only, Python 3.8+. Run from the ``student_resource/`` directory::

    python3 utils/validate_submission.py \
        --matching output/matching_results.tsv \
        --candidate output/candidate_pairs.tsv \
        --test-dir dataset/test

Exit code 0 means the files are safe to submit; 1 means fix the listed issues
(warnings never fail the run).

ID-existence check (off by default). By default the validator does NOT check that
every matched/candidate ID actually exists in the test set: that check loads all
Source-2/3 IDs into memory, which on the full ~1.7M-entity test set costs a few GB
(more when ``candidate_pairs.tsv`` is included). The default run therefore stays fast
and light and verifies every other rule; it prints a warning noting the check was
skipped. Pass ``--check-ids`` to turn it on (it reads ``test_source2.tsv`` /
``test_source3.tsv`` from ``--test-dir``); a missing/garbage matched ID only lowers
your score rather than being rejected by the scorer, so this check is a diagnostic,
not a gate. If ``--check-ids`` runs out of memory, drop ``--candidate`` (the candidate
cross-check is the biggest memory user, and the matching file is the only one scored).
"""

import argparse
import os
import sys

DELIM = "\t"
MAX_EXAMPLES = 5  # how many offending IDs to show per issue
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]


def read_ids(path):
    """Return the set of first-column entity IDs from a source TSV.

    The header row is skipped and blank lines are ignored.
    """
    with open(path, encoding="utf-8") as f:
        next(f, None)  # skip header
        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}


def examples(items):
    """Return a short, human-readable sample of ``items`` for an error message."""
    items = sorted(items)
    shown = ", ".join(items[:MAX_EXAMPLES])
    if len(items) > MAX_EXAMPLES:
        return f"{len(items)} total, e.g. {shown}, ..."
    return shown


def load_match_targets(test_dir, warnings):
    """Return the set of valid S2/S3 match IDs, or ``None`` if unavailable.

    Only called when ``--check-ids`` is on. When ``test_source2.tsv`` or
    ``test_source3.tsv`` is missing we cannot check that matched IDs exist, so we
    record a warning and return ``None`` to signal that the existence check should be
    skipped.
    """
    targets = set()
    for name in ("test_source2.tsv", "test_source3.tsv"):
        path = os.path.join(test_dir, name)
        if not os.path.isfile(path):
            warnings.append(
                f"{path} not found — skipping the (optional) check that matched "
                f"IDs exist in the test set. Every other rule is still checked. "
                f"This is the lighter-memory mode; provide test_source2/3.tsv to "
                f"enable the ID-existence check."
            )
            return None
        targets |= read_ids(path)
    return targets


def validate_id_list_file(path, expected_header, col_label, required, valid_ids, errors):
    """Validate one results-style TSV (matching or candidate).

    Applies the shared formatting rules and appends any problems to ``errors``.
    Returns a ``{source1_id: set(matched/candidate ids)}`` mapping, or ``None`` on a
    fatal problem (missing file, empty file, or a broken header) that stops parsing.
    """
    if not os.path.isfile(path):
        errors.append(f"File not found: {path}")
        return None

    name = os.path.basename(path)
    mapping = {}
    seen, dup_rows, intra_dupes = set(), set(), set()
    self_matches, wrong_prefix, unknown = set(), set(), set()
    n_rows = empties = 0

    with open(path, encoding="utf-8") as f:
        header = f.readline()
        if not header:
            errors.append(f"{name} is empty.")
            return None
        if DELIM not in header and "," in header:  # the #1 mistake: a CSV
            errors.append(
                f"{name}: header has no TAB but contains commas — the file looks "
                "COMMA-separated. Submissions must be TAB-separated (.tsv); "
                "write it with df.to_csv(sep='\\t', index=False)."
            )
            return None
        cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
        if cols != expected_header:
            errors.append(
                f"{name}: unexpected header {cols}. "
                f"Expected exactly {expected_header} (tab-separated)."
            )
            return None

        for line_num, line in enumerate(f, start=2):
            s1, tab, rest = line.partition(DELIM)
            if not tab:
                if s1.strip():
                    errors.append(
                        f"{name}: malformed row (no tab) at line {line_num}: "
                        f"{line.rstrip()!r}"
                    )
                continue

            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)

            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if not ids:
                empties += 1
                mapping[s1] = set()
                continue
            if len(ids) != len(set(ids)):
                intra_dupes.add(s1)
            id_set = set(ids)
            mapping[s1] = id_set
            for mid in id_set:
                if mid.startswith("S1-"):
                    self_matches.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_prefix.add(mid)
                elif valid_ids is not None and mid not in valid_ids:
                    unknown.add(mid)

    # Aggregate the per-category findings. Each entry is (offenders, message);
    # only non-empty categories become errors.
    findings = [
        (
            dup_rows,
            "{name}: duplicate source1_entity_id row(s): {ex}. "
            "Each S1 entity may appear on only one row.",
        ),
        (
            intra_dupes,
            "{name}: repeated ID inside a {col} list for: {ex}. "
            "No duplicate IDs are allowed within a list.",
        ),
        (
            self_matches,
            "{name}: {col} contains Source-1 IDs (self-matches): {ex}. "
            "Only S2-/S3- IDs are allowed.",
        ),
        (
            wrong_prefix,
            "{name}: {col} contains IDs without an S2-/S3- prefix: {ex}.",
        ),
        (
            unknown,
            "{name}: {col} references IDs not in the test "
            "Source-2/3 files: {ex}.",
        ),
        (
            required - seen,
            "{name}: required S1 entity(ies) missing: {ex}. "
            "Every entity in test_source1.tsv needs a row (empty = no match).",
        ),
        (
            seen - required,
            "{name}: row(s) using an S1 ID that is not in the test set: {ex}.",
        ),
    ]
    for offenders, message in findings:
        if offenders:
            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))

    print(f"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).")
    return mapping


def validate(matching_path, candidate_path, test_dir, check_ids=False):
    """Validate the submission output(s); return ``(errors, warnings)`` lists.

    ``check_ids`` (``--check-ids``) turns on the optional, memory-heavy check that
    every matched/candidate ID exists in the test Source-2/3 files. It is off by
    default so the common run stays fast and light.
    """
    errors, warnings = [], []

    source1 = os.path.join(test_dir, "test_source1.tsv")
    if not os.path.isfile(source1):
        errors.append(f"Test source1 file not found: {source1} (check --test-dir).")
        return errors, warnings
    required = read_ids(source1)
    print(f"  required S1 entities: {len(required)}")

    if check_ids:
        valid_ids = load_match_targets(test_dir, warnings)
        if valid_ids is not None:
            print(f"  valid S2/S3 match IDs: {len(valid_ids)}")
    else:
        valid_ids = None
        warnings.append(
            "ID-existence check is OFF (the default) — not checking that matched/"
            "candidate IDs exist in the test set. Every other rule is still checked. "
            "Re-run with --check-ids to enable it (needs test_source2/3.tsv; uses "
            "more memory). A nonexistent ID only lowers your score, never rejects "
            "your submission."
        )

    matched = validate_id_list_file(
        matching_path, MATCHING_HEADER, "matched_entity_ids", required, valid_ids, errors
    )

    # candidate_pairs.tsv is optional: if it's absent we skip its checks with a
    # warning (it's still expected in your final submission zip). A missing
    # candidate file never fails this run on its own.
    candidate = None
    if candidate_path and os.path.isfile(candidate_path):
        candidate = validate_id_list_file(
            candidate_path, CANDIDATE_HEADER, "candidate_entity_ids",
            required, valid_ids, errors,
        )
    elif candidate_path:
        warnings.append(
            f"{candidate_path} not found — skipping candidate_pairs.tsv checks. "
            "It is optional here, but your final submission zip must include "
            "output/candidate_pairs.tsv."
        )

    # Soft check: your final matches should come from your blocking candidates.
    # A matched ID absent from candidate_pairs.tsv usually means a pipeline bug,
    # so we warn but never fail on it.
    if matched is not None and candidate is not None:
        offenders = {
            s1 for s1, mids in matched.items() if mids - candidate.get(s1, set())
        }
        if offenders:
            warnings.append(
                f"{len(offenders)} S1 entity(ies) have matched IDs not present in "
                f"candidate_pairs.tsv, e.g. {examples(offenders)}. Final matches "
                "normally come from your blocking candidates — double-check these."
            )

    return errors, warnings


def main():
    parser = argparse.ArgumentParser(
        description="Validate ML Challenge 2026 submission output files before submitting."
    )
    parser.add_argument(
        "--matching",
        "-m",
        default="output/matching_results.tsv",
        help="Path to matching_results.tsv (default: %(default)s)",
    )
    parser.add_argument(
        "--candidate",
        "-c",
        default=None,
        help="Path to candidate_pairs.tsv "
        "(default: output/candidate_pairs.tsv if it exists).",
    )
    parser.add_argument(
        "--test-dir",
        "-t",
        default="dataset/test",
        help="Folder with test_source1/2/3.tsv (default: %(default)s). "
        "test_source2/3.tsv are only read when --check-ids is given.",
    )
    parser.add_argument(
        "--check-ids",
        action="store_true",
        help="Also check that every matched/candidate ID exists in the test "
        "Source-2/3 files. Off by default (loads all S2/S3 IDs into memory — a few "
        "GB on the full test set). A nonexistent ID only lowers your score, so this "
        "is a diagnostic, not a submission gate.",
    )
    args = parser.parse_args()

    # candidate_pairs.tsv is optional; default to the conventional path and let
    # validate() skip (with a warning) if the file isn't there.
    candidate_path = args.candidate or "output/candidate_pairs.tsv"

    print("ML Challenge 2026 — submission validator")
    print(f"  test dir: {args.test_dir}")
    try:
        errors, warnings = validate(
            args.matching, candidate_path, args.test_dir, check_ids=args.check_ids
        )
    except UnicodeDecodeError:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(
            f"  1. A file is not valid UTF-8 text (most likely {args.matching} or "
            f"{candidate_path}). Re-save it as a plain UTF-8, tab-separated .tsv — "
            "not cp1252/Latin-1, and not a compressed or binary file (.gz/.xlsx/"
            ".parquet) renamed to .tsv. In pandas: "
            "df.to_csv(path, sep='\\t', index=False, encoding='utf-8')."
        )
        return 1
    except OSError as exc:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(f"  1. Could not read a file: {exc}.")
        return 1

    print()
    for warning in warnings:
        print(f"WARNING: {warning}")
    if errors:
        print(f"FAIL — {len(errors)} issue(s) to fix before submitting:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        return 1
    print("PASS — no blocking issues found. Safe to submit.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
import sys
sys.path.insert(0, '/tmp/src')
sys.argv = ['cross_encoder_score.py', '--input', '/kaggle/input', '--ce-dir', '/kaggle/input', '--v4-dir', '/kaggle/input', '--out-dir', '/kaggle/working/ce_blend', '--validator', '/tmp/src/validate_submission.py']

In [ ]:
"""Cross-encoder re-scoring of borderline pairs (experiment; branch cross-encoder).

Model: distilbert-base-multilingual-cased (Apache-2.0), fine-tuned by cross_encoder_train.py as a binary pair
classifier. Input: "{business_name} | {business_address} | {country}" of the S1 record [SEP] the same for the
candidate, raw organiser text, max 128 tokens.
Blend (borderline pairs only, BAND[0] < p_model < BAND[1]): p = (1 - w) * p_model + w * p_ce; other pairs keep
p_model. Then the usual selection (threshold, margin, one owner per S2/S3).

Test usage (GPU), after run_model_v4.py and cross_encoder_train.py:
  python src/cross_encoder_score.py --input DIR --ce-dir DIR --v4-dir DIR --out-dir output/ce_blend [--validator PATH]
The blend is re-tuned on model v4's validation scores (w and threshold on the tune half, margin and one owner as
v4; report half = check), so it matches the model whose test scores it re-scores. GO only if the report half beats
w = 0 overall AND for US and India with precision >= 0.99; otherwise w = 0 (the v4 scores unchanged). Countries
not in train get v4's unseen-country threshold shift (0 unless v4's simulation kept it).
"""
from __future__ import annotations

import argparse
import json
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import polars as pl

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:
    sys.path.insert(0, "/tmp/src")

MODEL = "distilbert-base-multilingual-cased"
MAX_LEN = 128
BAND = (0.02, 0.98)
T0 = time.time()


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - T0:6.0f}s] {msg}", flush=True)


def find(root: Path, name: str, not_parent: tuple = ("normalised", "features_extra", "aug")) -> Path:
    hits = sorted(p for p in root.rglob(name) if p.parent.name not in not_parent)
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits[0]


def texts(root: Path, side: str, ids: pl.Series) -> dict[str, str]:
    """entity_id -> "name | address | country" from the organiser files of one side (train / test)."""
    out = {}
    for k in (1, 2, 3):
        d = (pl.scan_parquet(find(root, f"{side}_source{k}.parquet")).select("entity_id", "business_name", "business_address", "country")
             .filter(pl.col("entity_id").is_in(ids.implode())).collect())
        t = d.select("entity_id", pl.concat_str([pl.col("business_name").fill_null(""), pl.col("business_address").fill_null(""),
                                                 pl.col("country").fill_null("")], separator=" | ").alias("t"))
        out.update(zip(t["entity_id"].to_list(), t["t"].to_list()))
    return out


def score_pairs(model, tok, pairs: pl.DataFrame, txt: dict, batch: int = 512, log_every: int = 200) -> np.ndarray:
    import torch
    model.eval()
    a, b = [txt.get(s, "") for s in pairs["s1"].to_list()], [txt.get(c, "") for c in pairs["cand"].to_list()]
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(a), batch):
            enc = tok(a[i:i + batch], b[i:i + batch], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to("cuda")
            out.append(torch.sigmoid(model(**enc).logits.float().squeeze(-1)).cpu().numpy())
            if log_every and (i // batch) % log_every == 0:
                log(f"  scored {i + len(a[i:i + batch]):,} / {len(a):,}")
    return np.concatenate(out) if out else np.empty(0)


def blend(p_model: np.ndarray, p_ce: np.ndarray, w: float) -> np.ndarray:
    """p_ce: NaN outside the borderline band."""
    border = ~np.isnan(p_ce)
    p = p_model.copy()
    p[border] = (1 - w) * p_model[border] + w * p_ce[border]
    return p


WS = [0.0, 0.35, 0.5, 0.65, 0.8, 0.9, 1.0]


def main() -> None:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from selection import f05_table, select, summary
    from submission import write_submission
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, type=Path)
    ap.add_argument("--ce-dir", required=True, type=Path, help="folder holding ce_model/ (searched)")
    ap.add_argument("--v4-dir", required=True, type=Path, help="folder holding model v4's test_scores / val_scores / model_config (searched)")
    ap.add_argument("--out-dir", required=True, type=Path)
    ap.add_argument("--validator", default=None)
    a = ap.parse_args()
    a.out_dir.mkdir(parents=True, exist_ok=True)
    mdir = next(p for p in a.ce_dir.rglob("config.json") if p.parent.name == "ce_model").parent
    v4cfg = json.loads(next(a.v4_dir.rglob("model_config.json")).read_text())
    tok = AutoTokenizer.from_pretrained(mdir)
    model = AutoModelForSequenceClassification.from_pretrained(mdir).to("cuda")
    rep = ["# Cross-encoder blend on model v4\n", f"v4: selection {v4cfg['selection']}, unseen shift {v4cfg['unseen_tau_shift']}, "
           f"pseudo-label {v4cfg['pseudo_label']}, cross features {v4cfg['use_cross']}.\n"]

    # ---- validation: re-tune the blend on v4's validation scores
    vs = pl.read_parquet(next(a.v4_dir.rglob("val_scores.parquet"))).select("s1", "cand", "p")
    split = pl.read_parquet(find(a.input, "g25_split.parquet"))
    val_s1 = split.filter(pl.col("role") == "val").select(pl.col("source1_entity_id").alias("s1")).join(
        pl.read_parquet(find(a.input, "train_source1.parquet"), columns=["entity_id", "country"]).rename({"entity_id": "s1"}), on="s1", how="left")
    gt = pl.read_parquet(find(a.input, "train_ground_truth.parquet"))
    truth = (gt.join(val_s1, left_on="source1_entity_id", right_on="s1", how="semi")
             .select(pl.col("source1_entity_id").alias("s1"), pl.col("matched_entity_ids").str.split(",").alias("cand"))
             .explode("cand", empty_as_null=True).filter(pl.col("cand").is_not_null() & (pl.col("cand") != "")))
    half = pl.col("s1").str.extract(r"(\d+)$").cast(pl.Int64) % 2
    s1_tune, s1_rep = val_s1.filter(half == 0), val_s1.filter(half == 1)
    vb = vs.filter((pl.col("p") > BAND[0]) & (pl.col("p") < BAND[1]))
    vtxt = texts(a.input, "train", pl.concat([vb["s1"], vb["cand"]]).unique())
    p_ce_v = score_pairs(model, tok, vb.select("s1", "cand"), vtxt, log_every=0)
    log(f"validation borderline pairs scored: {vb.height:,}")
    full = vs.join(vb.select("s1", "cand").with_columns(pl.Series("p_ce", p_ce_v)), on=["s1", "cand"], how="left")
    pm, pc = full["p"].to_numpy(), full["p_ce"].fill_null(np.nan).to_numpy()
    cur = {"margin": v4cfg["selection"]["margin"], "one_owner": True}
    res = []
    for w in WS:
        sv = full.select("s1", "cand").with_columns(pl.Series("p", blend(pm, pc, w)))
        grid = [(f05_table(select(sv, tau=t, **cur), truth, s1_tune)["f05"].mean(), t) for t in [round(x, 3) for x in np.arange(0.30, 0.91, 0.025)]]
        f_t, tau = max(grid)
        sm = summary(f05_table(select(sv, tau=tau, **cur), truth, s1_rep), by="country")
        gg = {r["group"]: r for r in sm.iter_rows(named=True)}
        res.append({"w": w, "tau": tau, "tune_f05": f_t, **{k: gg[k]["macro_f05"] for k in gg}, "precision": gg["overall"]["precision"],
                    "recall": gg["overall"]["recall"]})
        log(f"w {w}: {res[-1]}")
    res = pl.DataFrame(res)
    base = res.filter(pl.col("w") == 0.0).row(0, named=True)
    best = res.filter(pl.col("w") > 0).sort("tune_f05", descending=True).row(0, named=True)
    go = (best["overall"] > base["overall"] and best["country=US"] > base["country=US"] and best["country=India"] > base["country=India"]
          and best["precision"] >= 0.99)
    ch = best if go else base
    rep += ["## Validation (report half; tau tuned on the tune half)\n", str(res) + "\n",
            f"**GO = {go}**: w = {ch['w']}, tau = {ch['tau']} (report {ch['overall']:.4f} vs w=0 {base['overall']:.4f})\n"]
    log(f"GO = {go}; using w {ch['w']} tau {ch['tau']}")

    # ---- test
    st = pl.read_parquet(next(a.v4_dir.rglob("test_scores.parquet")))
    s1_all = pl.read_parquet(find(a.input, "test_source1.parquet"), columns=["entity_id", "country"]).rename({"entity_id": "s1"})
    if ch["w"] > 0:
        border = st.filter((pl.col("p") > BAND[0]) & (pl.col("p") < BAND[1]))
        txt = texts(a.input, "test", pl.concat([border["s1"], border["cand"]]).unique())
        t0 = time.time()
        p_ce = score_pairs(model, tok, border.select("s1", "cand"), txt)
        log(f"scored {border.height:,} borderline test pairs in {time.time() - t0:.0f}s")
        st = st.join(border.select("s1", "cand").with_columns(pl.Series("p_ce", p_ce)), on=["s1", "cand"], how="left")
    else:
        st = st.with_columns(pl.lit(None, dtype=pl.Float64).alias("p_ce"))
    st = st.with_columns(pl.Series("p_blend", blend(st["p"].to_numpy(), st["p_ce"].fill_null(np.nan).to_numpy(), ch["w"])))
    st.write_parquet(a.out_dir / "test_scores_blend.parquet")
    unseen = ~pl.col("country").is_in(v4cfg["train_countries"])
    tau_row = (pl.lit(ch["tau"]) + pl.when(unseen).then(pl.lit(v4cfg["unseen_tau_shift"])).otherwise(0.0)).clip(0.3, 0.9)
    x = st.join(s1_all, on="s1", how="left").filter(pl.col("p_blend") >= tau_row)
    matched = select(x.select("s1", "cand", pl.col("p_blend").alias("p")), tau=0.0, **cur)
    cand = pl.read_parquet(find(a.input, "test_candidates.parquet"), columns=["s1", "cand", "p_u50"])
    lists = cand.sort(["s1", "p_u50", "cand"], descending=[False, True, False], nulls_last=True).group_by("s1", maintain_order=True).agg(pl.col("cand"))
    ml = matched.sort(["s1", "p"], descending=[False, True]).group_by("s1", maintain_order=True).agg(pl.col("cand"))
    write_submission(a.out_dir, s1_all["s1"].to_list(), dict(zip(ml["s1"].to_list(), ml["cand"].to_list())),
                     dict(zip(lists["s1"].to_list(), lists["cand"].to_list())))
    rep.append(f"Test: {matched.height:,} matches.\n")
    if a.validator:
        import importlib.util
        spec = importlib.util.spec_from_file_location("vs", a.validator); vs_ = importlib.util.module_from_spec(spec); spec.loader.exec_module(vs_)
        with tempfile.TemporaryDirectory() as td:
            for k in (1, 2, 3):
                pl.read_parquet(find(a.input, f"test_source{k}.parquet"), columns=["entity_id"]).write_csv(Path(td) / f"test_source{k}.tsv", separator="\t")
            e, w = vs_.validate(str(a.out_dir / "matching_results.tsv"), str(a.out_dir / "candidate_pairs.tsv"), td, check_ids=True)
        rep.append(f"Validator (--check-ids): {len(e)} errors, {len(w)} warnings {e[:3]}\n")
        log(f"validator: {len(e)} errors, {len(w)} warnings")
    n = s1_all.join(matched.group_by("s1").len("n"), on="s1", how="left").with_columns(pl.col("n").fill_null(0))
    pcn = n.group_by("country").agg(pl.col("n").mean().alias("matches_per_s1"), (pl.col("n") == 0).mean().alias("share_no_match")).sort("country")
    rep += ["## Matches per S1\n", str(pcn) + "\n"]
    (a.out_dir / "ce_test_report.md").write_text("\n".join(rep))
    log("done")


if __name__ == "__main__":
    main()
